## Step 1 - Imports and Setup
Load required libraries and set the random seed for reproducible model experiments.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import randint

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

RANDOM_STATE = 42

## Step 2 - Load House Feature Matrix
Read engineered house features and verify shape, columns, and missing values.

In [2]:
X_df = pd.read_csv('../../data/house/satilir_properties_house_feature_engineered.csv')
print('Shape:', X_df.shape)
print('Columns:', len(X_df.columns))
print('Missing values:', int(X_df.isna().sum().sum()))
X_df.head()

Shape: (1173, 40)
Columns: 40
Missing values: 0


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,...,cat_ohe_mid__address_part_2_None,cat_ohe_mid__address_part_2_infrequent_sklearn,num__rooms,num__free_land_ratio,num__feature_score,num__log_area_m2,num__log_land_area_sot,num__log_floor,num__log_area_per_room,num__log_rooms_per_floor
0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,1.0,0.0,3.0,4.117647e-01,0.526316,4.615121,0.993252,0.693147,3.536116,1.386294
1,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,1.0,0.0,3.0,5.000000e-01,0.631579,4.709530,1.163151,0.693147,3.628775,1.386294
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,1.0,1.666667e-08,0.473684,4.110874,0.470004,0.693147,4.110873,0.693147
3,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,6.0,4.000000e-09,0.842105,5.525453,1.252763,1.386294,3.753418,1.098612
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,6.0,4.882498e-01,0.263158,5.326419,1.609438,1.098612,3.558676,1.386294


## Step 3 - Load Target Vector
Load the log-transformed house target values and check data integrity.

In [3]:
y_df = pd.read_csv('../../data/house/satilir_properties_house_target_log.csv')
print('Shape:', y_df.shape)
print('Columns:', len(y_df.columns))
print('Missing values:', int(y_df.isna().sum().sum()))
y_df.head()

Shape: (1173, 1)
Columns: 1
Missing values: 0


,price_log1p
0,11.736077
1,11.884496
2,10.463132
3,12.538971
4,11.225257


## Step 4 - Prepare Modeling Split
Clean target values, create modeling datasets, and build the stratified train/test split.

In [4]:
# Train and compare stronger tree ensembles on X_df / y_df
# Optimize for raw-scale MAE (after reversing log1p) since this reflects real price error.

target_col = 'price_log1p' if 'price_log1p' in y_df.columns else y_df.columns[0]
X_model = X_df.copy()
y_model = pd.to_numeric(y_df[target_col], errors='coerce')

valid_mask = y_model.notna()
X_model = X_model.loc[valid_mask].reset_index(drop=True)
y_model = y_model.loc[valid_mask].reset_index(drop=True)

# Stratified split over target quantiles for a more stable holdout
y_bins = pd.qcut(y_model, q=10, labels=False, duplicates='drop')
stratify_arg = y_bins if pd.Series(y_bins).nunique() > 1 else None

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_arg,
)

## Step 5 - Tune Core Tree Models
Run randomized search for RandomForest and ExtraTrees and compare raw-scale holdout metrics.

In [5]:
def mae_raw_from_log(y_true_log, y_pred_log):
    y_true_raw = np.expm1(np.asarray(y_true_log))
    y_pred_raw = np.expm1(np.clip(np.asarray(y_pred_log), 0, None))
    return mean_absolute_error(y_true_raw, y_pred_raw)

raw_mae_scorer = make_scorer(mae_raw_from_log, greater_is_better=False)

search_spaces = {
    'RandomForest': (
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'n_estimators': randint(1400, 3200),
            'max_depth': [None, 16, 20, 24, 32],
            'max_features': [0.6, 0.7, 0.8, 0.9],
            'min_samples_split': randint(3, 10),
            'min_samples_leaf': randint(1, 4),
            'bootstrap': [True],
        },
    ),
    'ExtraTrees': (
        ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'n_estimators': randint(1400, 3200),
            'max_depth': [None, 16, 20, 24, 32, 40],
            'max_features': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'min_samples_split': randint(2, 10),
            'min_samples_leaf': randint(1, 4),
            'bootstrap': [False],
            'criterion': ['squared_error', 'absolute_error'],
        },
    ),
}

results = []
best_model_name = None
best_model = None
best_model_params = None
best_test_mae = np.inf

y_test_raw = np.expm1(y_test.to_numpy())

for model_name, (estimator, param_dist) in search_spaces.items():
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=25,
        scoring=raw_mae_scorer,
        cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )

    search.fit(X_train, y_train)
    pred_log = np.clip(search.best_estimator_.predict(X_test), 0, None)
    pred_raw = np.expm1(pred_log)

    test_mae = mean_absolute_error(y_test_raw, pred_raw)
    test_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_raw))
    test_r2 = r2_score(y_test_raw, pred_raw)

    results.append(
        {
            'model': model_name,
            'cv_mae_raw': -search.best_score_,
            'test_mae_raw': test_mae,
            'test_rmse_raw': test_rmse,
            'test_r2_raw': test_r2,
            'best_params': search.best_params_,
        }
    )

    if test_mae < best_test_mae:
        best_test_mae = test_mae
        best_model_name = model_name
        best_model = search.best_estimator_
        best_model_params = search.best_params_

results_df = pd.DataFrame(results).sort_values('test_mae_raw').reset_index(drop=True)
summary_cols = ['model', 'cv_mae_raw', 'test_mae_raw', 'test_rmse_raw', 'test_r2_raw']

print('--- Model Comparison (raw-scale) ---')
print(results_df[summary_cols].round(4))

print('\n--- Best Model ---')
print('Model:', best_model_name)
print(f"Test MAE (raw): ${best_test_mae:,.0f}")
print(
    f"Test RMSE (raw): ${results_df.loc[results_df['model'] == best_model_name, 'test_rmse_raw'].iloc[0]:,.0f}"
)
print(
    'Test R2 (raw):',
    round(results_df.loc[results_df['model'] == best_model_name, 'test_r2_raw'].iloc[0], 4),
)
print('Best Params:', best_model_params)
print('Feature count:', X_train.shape[1])

best_model_results = results_df

--- Model Comparison (raw-scale) ---
          model  cv_mae_raw  test_mae_raw  test_rmse_raw  test_r2_raw
0    ExtraTrees  49027.7459    53211.6186    133557.3751       0.7523
1  RandomForest  54819.5365    58810.0720    135628.0919       0.7446

--- Best Model ---
Model: ExtraTrees
Test MAE (raw): $53,212
Test RMSE (raw): $133,557
Test R2 (raw): 0.7523
Best Params: {'bootstrap': False, 'criterion': 'absolute_error', 'max_depth': None, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 2580}
Feature count: 40


## Step 6 - Compare Final Candidates
Evaluate RF, ET, and LightGBM candidates on the holdout set and append trial proof history.

In [6]:
from datetime import datetime, timezone
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from lightgbm import LGBMRegressor




def get_params_from_results(model_name, defaults):
    if 'results_df' in globals() and isinstance(results_df, pd.DataFrame):
        if {'model', 'best_params'}.issubset(results_df.columns):
            row = results_df.loc[results_df['model'] == model_name]
            if not row.empty and isinstance(row.iloc[0]['best_params'], dict):
                return row.iloc[0]['best_params']

    if 'best_model_results' in globals() and isinstance(best_model_results, pd.DataFrame):
        if {'model', 'best_params'}.issubset(best_model_results.columns):
            row = best_model_results.loc[best_model_results['model'] == model_name]
            if not row.empty and isinstance(row.iloc[0]['best_params'], dict):
                return row.iloc[0]['best_params']

    return defaults


rf_defaults = {
    'n_estimators': 2500,
    'max_depth': 16,
    'max_features': 0.6,
    'min_samples_split': 3,
    'min_samples_leaf': 1,
    'bootstrap': True,
}
et_defaults = {
    'n_estimators': 2600,
    'max_depth': 20,
    'max_features': 0.7,
    'min_samples_split': 4,
    'min_samples_leaf': 1,
    'bootstrap': False,
    'criterion': 'squared_error',
}

rf_params = get_params_from_results('RandomForest', rf_defaults)
et_params = get_params_from_results('ExtraTrees', et_defaults)

models = {
    'rf': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **rf_params),
    'et': ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1, **et_params),
    'lgbm': LGBMRegressor(
        n_estimators=1800,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
}

rows = []
y_test_raw = np.expm1(y_test.to_numpy())

for name, model in models.items():
    m = clone(model)
    m.fit(X_train, y_train)

    pred_log = np.clip(m.predict(X_test), 0, None)
    pred_raw = np.expm1(pred_log)

    rows.append(
        {
            'trial_name': f'{name}',
            'test_r2_raw': r2_score(y_test_raw, pred_raw),
            'test_rmse_raw': np.sqrt(mean_squared_error(y_test_raw, pred_raw)),
            'test_mae_raw': mean_absolute_error(y_test_raw, pred_raw),
            'best_params': str(m.get_params()),
        }
    )

model_compare_df = pd.DataFrame(rows).sort_values('test_mae_raw').reset_index(drop=True)
best_row = model_compare_df.iloc[0]

print('--- Model Comparison ---')
print(model_compare_df[['trial_name', 'test_r2_raw', 'test_rmse_raw', 'test_mae_raw']].round(4))

print('\n--- Best  ---')
print('Model:', best_row['trial_name'])
print(f"Test MAE (raw): ${best_row['test_mae_raw']:,.0f}")
print(f"Test RMSE (raw): ${best_row['test_rmse_raw']:,.0f}")
print('Test R2 (raw):', round(float(best_row['test_r2_raw']), 4))

# Save proof rows
proof_path = Path('../../models/metrics/house_model_trial_proof.csv')
proof_path.parent.mkdir(parents=True, exist_ok=True)
timestamp_utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

proof_rows = model_compare_df.copy()
proof_rows.insert(0, 'timestamp_utc', timestamp_utc)
proof_rows.insert(2, 'source', 'cell_5_model_compare_no_oof')
proof_rows.insert(3, 'objective', 'minimize_raw_mae')
proof_rows.insert(4, 'cv_primary_score', np.nan)
proof_rows = proof_rows[
    ['timestamp_utc', 'trial_name', 'source', 'objective', 'cv_primary_score', 'test_r2_raw', 'test_rmse_raw', 'test_mae_raw', 'best_params']
]

if proof_path.exists():
    existing = pd.read_csv(proof_path)
    proof_history = pd.concat([existing, proof_rows], ignore_index=True)
else:
    proof_history = proof_rows

proof_history.to_csv(proof_path, index=False)
print(f'Saved trial proof history: {proof_path}')

--- Model Comparison ---
  trial_name  test_r2_raw  test_rmse_raw  test_mae_raw
0         et       0.7523    133557.3751    53211.6186
1         rf       0.7446    135628.0919    58810.0720
2       lgbm       0.7373    137534.8453    62240.7216

--- Best  ---
Model: et
Test MAE (raw): $53,212
Test RMSE (raw): $133,557
Test R2 (raw): 0.7523
Saved trial proof history: ..\..\models\metrics\house_model_trial_proof.csv


## Step 7 - Stability Across Seeds
Run a multi-seed robustness check for the selected model and store stability proof metrics.

In [7]:
# 10-seed stability check for the best no-OOF model (raw scale)
from sklearn.base import clone

# Determine selected model from previous no-OOF comparison cell
if 'model_compare_df' in globals() and isinstance(model_compare_df, pd.DataFrame) and not model_compare_df.empty:
    selected_trial = str(model_compare_df.iloc[0]['trial_name'])
    selected_model_key = selected_trial.replace('no_oof_', '')
elif 'best_row' in globals() and isinstance(best_row, pd.Series) and 'trial_name' in best_row:
    selected_trial = str(best_row['trial_name'])
    selected_model_key = selected_trial.replace('no_oof_', '')
else:
    selected_model_key = 'rf'

# Rebuild model dictionary if needed
if 'models' not in globals() or selected_model_key not in models:
    rf_defaults = {
        'n_estimators': 2500,
        'max_depth': 16,
        'max_features': 0.6,
        'min_samples_split': 3,
        'min_samples_leaf': 1,
        'bootstrap': True,
    }
    et_defaults = {
        'n_estimators': 2600,
        'max_depth': 20,
        'max_features': 0.7,
        'min_samples_split': 4,
        'min_samples_leaf': 1,
        'bootstrap': False,
        'criterion': 'squared_error',
    }

    models = {
        'rf': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **rf_defaults),
        'et': ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1, **et_defaults),
        'lgbm': LGBMRegressor(
            n_estimators=1800,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.8,
            min_child_samples=20,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),
    }

if selected_model_key not in models:
    raise ValueError(f"Selected model '{selected_model_key}' is not available in models dictionary.")

# Ensure base data exists
if 'X_model' not in globals() or 'y_model' not in globals():
    target_col = 'price_log1p' if 'price_log1p' in y_df.columns else y_df.columns[0]
    X_model = X_df.copy()
    y_model = pd.to_numeric(y_df[target_col], errors='coerce')
    valid_mask = y_model.notna()
    X_model = X_model.loc[valid_mask].reset_index(drop=True)
    y_model = y_model.loc[valid_mask].reset_index(drop=True)

stability_rows = []

for s in range(1, 11):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_model,
        y_model,
        test_size=0.2,
        random_state=s,
    )

    model_s = clone(models[selected_model_key])
    if hasattr(model_s, 'random_state'):
        model_s.set_params(random_state=s)

    model_s.fit(X_tr, y_tr)

    pred_log_s = np.clip(model_s.predict(X_te), 0, None)
    pred_raw_s = np.expm1(pred_log_s)
    y_te_raw = np.expm1(y_te.to_numpy())

    stability_rows.append(
        {
            'seed': s,
            'trial_name': f'stability_{selected_model_key}_seed_{s}',
            'test_r2_raw': r2_score(y_te_raw, pred_raw_s),
            'test_rmse_raw': np.sqrt(mean_squared_error(y_te_raw, pred_raw_s)),
            'test_mae_raw': mean_absolute_error(y_te_raw, pred_raw_s),
        }
    )

stability_df = pd.DataFrame(stability_rows).sort_values('test_mae_raw').reset_index(drop=True)
display(stability_df)

print(f"\nNo-OOF stability summary for model: {selected_model_key}")
print(f"Mean Test MAE  : ${stability_df['test_mae_raw'].mean():,.0f}")
print(f"Median Test MAE: ${stability_df['test_mae_raw'].median():,.0f}")
print(f"Best Test MAE  : ${stability_df['test_mae_raw'].min():,.0f}")
print(f"Worst Test MAE : ${stability_df['test_mae_raw'].max():,.0f}")
print(f"Mean Test R2   : {stability_df['test_r2_raw'].mean():.4f}")

# Append stability proof rows
proof_path = Path('../../models/metrics/house_model_trial_proof.csv')
proof_path.parent.mkdir(parents=True, exist_ok=True)
timestamp_utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

proof_rows = stability_df.copy()
proof_rows.insert(0, 'timestamp_utc', timestamp_utc)
proof_rows.insert(2, 'source', 'cell_6_stability_no_oof')
proof_rows.insert(3, 'objective', 'seed_stability_raw')
proof_rows.insert(4, 'cv_primary_score', np.nan)
proof_rows['best_params'] = np.nan
proof_rows = proof_rows[
    ['timestamp_utc', 'trial_name', 'source', 'objective', 'cv_primary_score', 'test_r2_raw', 'test_rmse_raw', 'test_mae_raw', 'best_params']
]

if proof_path.exists():
    existing = pd.read_csv(proof_path)
    proof_history = pd.concat([existing, proof_rows], ignore_index=True)
else:
    proof_history = proof_rows

proof_history.to_csv(proof_path, index=False)
print(f"Saved stability proof history: {proof_path}")

,seed,trial_name,test_r2_raw,test_rmse_raw,test_mae_raw
0,7,stability_et_seed_7,0.885918,84166.048318,39784.447692
1,2,stability_et_seed_2,0.863295,88322.740744,41966.260167
2,4,stability_et_seed_4,0.854622,94665.996641,42001.498266
3,3,stability_et_seed_3,0.824708,113622.446945,43044.607057
4,1,stability_et_seed_1,0.849800,86310.462312,44010.646107
5,9,stability_et_seed_9,0.852794,94690.928551,45112.844677
6,8,stability_et_seed_8,0.847050,96711.776049,49146.574492
7,10,stability_et_seed_10,0.773983,139643.107320,50744.547326
8,5,stability_et_seed_5,0.547196,221425.488010,53310.717009
9,6,stability_et_seed_6,0.615189,223844.513675,66707.304173



No-OOF stability summary for model: et
Mean Test MAE  : $47,583
Median Test MAE: $44,562
Best Test MAE  : $39,784
Worst Test MAE : $66,707
Mean Test R2   : 0.7915
Saved stability proof history: ..\..\models\metrics\house_model_trial_proof.csv


## Step 8 - Save Final House Artifact
Fit the final house tabular model on full data and save model plus summary metadata.

In [8]:
# Save final tabular model artifact
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
from sklearn.base import clone

DATASET_NAME = 'house'
MODEL_OUTPUT_DIR = Path('../../models/tabular') / DATASET_NAME
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if 'X_model' not in globals() or 'y_model' not in globals():
    raise RuntimeError('Missing X_model/y_model. Run the training cells first.')

if 'models' not in globals() or not isinstance(models, dict) or len(models) == 0:
    raise RuntimeError('Missing models dictionary. Run the model setup cells first.')

if 'model_compare_df' in globals() and isinstance(model_compare_df, pd.DataFrame) and not model_compare_df.empty:
    selected_trial_name = str(model_compare_df.iloc[0]['trial_name'])
elif 'best_row' in globals() and isinstance(best_row, pd.Series) and 'trial_name' in best_row:
    selected_trial_name = str(best_row['trial_name'])
else:
    selected_trial_name = next(iter(models.keys()))

selected_model_key = selected_trial_name
for prefix in ('no_oof_', 'stability_', 'base_'):
    if selected_model_key.startswith(prefix):
        selected_model_key = selected_model_key.replace(prefix, '', 1)

if selected_model_key not in models:
    for candidate in models.keys():
        if candidate in selected_trial_name:
            selected_model_key = candidate
            break

if selected_model_key not in models:
    raise KeyError(f"Could not map trial '{selected_trial_name}' to available models: {list(models.keys())}")

final_model = clone(models[selected_model_key])
final_model.fit(X_model, y_model)

timestamp_utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
artifact_path = MODEL_OUTPUT_DIR / f'{DATASET_NAME}_tabular_model.joblib'
summary_path = MODEL_OUTPUT_DIR / f'{DATASET_NAME}_tabular_model_summary.json'

leaderboard_records = []
if 'model_compare_df' in globals() and isinstance(model_compare_df, pd.DataFrame) and not model_compare_df.empty:
    leaderboard_records = model_compare_df.to_dict(orient='records')

target_col = 'price_log1p'
if 'y_df' in globals() and isinstance(y_df, pd.DataFrame) and 'price_log1p' not in y_df.columns and len(y_df.columns) > 0:
    target_col = str(y_df.columns[0])

artifact = {
    'dataset': DATASET_NAME,
    'created_utc': timestamp_utc,
    'target_column': target_col,
    'target_transform': 'log1p',
    'selected_trial_name': selected_trial_name,
    'selected_model_key': selected_model_key,
    'feature_names': list(X_model.columns),
    'model': final_model,
    'leaderboard': leaderboard_records,
}
joblib.dump(artifact, artifact_path)

summary = {
    'dataset': DATASET_NAME,
    'created_utc': timestamp_utc,
    'artifact_path': str(artifact_path),
    'selected_trial_name': selected_trial_name,
    'selected_model_key': selected_model_key,
    'row_count': int(len(X_model)),
    'feature_count': int(X_model.shape[1]),
    'target_column': target_col,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved tabular model:', artifact_path)
print('Saved summary      :', summary_path)

Saved tabular model: ..\..\models\tabular\house\house_tabular_model.joblib
Saved summary      : ..\..\models\tabular\house\house_tabular_model_summary.json
